# Module 2 — Clinical Rotterdam Phenotype Classification (PCOS)
Revised to follow the Statement of Purpose (Sections 5.3 and 5.5) and to fix the issues found in the first version.

**Pipeline:** load data → derive Rotterdam phenotypes (proxy criteria) → feature engineering → leakage-safe feature set →
5-fold stratified CV (repeated) with median-impute / |Z|>3 cap / StandardScaler / SMOTE *inside* each fold →
XGBoost vs LightGBM vs Random Forest (vs TabNet) → macro-F1, weighted-F1, Cohen's kappa, MCC, per-class report →
TreeSHAP (global + per-phenotype waterfall) → phenotype-to-treatment mapping.

## 0. Setup (Google Colab)
Run this cell first. Colab already has pandas / scikit-learn / xgboost / lightgbm; the rest are installed here.
If Colab asks you to **restart the runtime** after installing, do so once and then run all cells again.

In [ ]:
!pip install -q shap imbalanced-learn pytorch-tabnet openpyxl

In [ ]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load and clean the Kaggle clinical workbook

In [ ]:
import glob

# Option A (simplest): run this cell and upload PCOS_data_without_infertility.xlsx when prompted.
#   Download it from Kaggle: "Polycystic Ovary Syndrome (PCOS)" by Prasoon Kottarathil.
# Option B: put the file in Google Drive, mount Drive, and set DATA_PATH below, e.g.
#   from google.colab import drive; drive.mount("/content/drive")
#   DATA_PATH = "/content/drive/MyDrive/PCOS_data_without_infertility.xlsx"
DATA_PATH = None

def find_workbook():
    hits = []
    for root in ["/content", "."]:
        hits += glob.glob(os.path.join(root, "**", "*.xls*"), recursive=True)
    hits = [h for h in hits if "sample_data" not in h]
    preferred = [h for h in hits if "without_infertility" in os.path.basename(h).lower()]
    return (preferred or hits or [None])[0]

full_path = DATA_PATH or find_workbook()
if full_path is None:
    try:
        from google.colab import files
        print("Upload PCOS_data_without_infertility.xlsx ...")
        uploaded = files.upload()
        full_path = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError("Workbook not found. Set DATA_PATH to the .xlsx file.")
print("Using file:", full_path)

xls = pd.ExcelFile(full_path)
sheet = "Full_new" if "Full_new" in xls.sheet_names else xls.sheet_names[-1]
df = pd.read_excel(xls, sheet_name=sheet)
print("Sheet:", sheet, "| original shape:", df.shape)

df.columns = df.columns.astype(str).str.strip()
df = df.replace("?", np.nan)

# Drop administrative columns and the dataset's own PCOS label (we derive phenotypes instead)
drop_patterns = r"^unnamed|sl\.?\s*no|patient file|pcos \(y/n\)|pregnan|blood group"
df = df.drop(columns=[c for c in df.columns if re.search(drop_patterns, c.lower())], errors="ignore")

for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(axis=1, how="all")
print("Cleaned shape:", df.shape)

## 2. Rotterdam phenotype derivation (proxy criteria)

**Documented deviations from the original SOP text (state these in the thesis):**
1. *HA proxy* – the SOP rule "hirsutism≥2 OR acne≥2 OR skin darkening=1" cannot be applied: all three columns are binary (0/1).
   Primary rule = hirsutism (`hair growth`) only. A broader rule (hirsutism OR acne OR skin darkening) is reported as a sensitivity analysis.
2. *OD proxy* – `Cycle length(days)` only spans 0–12, so it is almost certainly period duration, not cycle length; a ">35 days" rule can never fire.
   It is auto-disabled by `validate_config`; OD comes from `Cycle(R/I)` (4 = irregular, 2 = regular; any other code = unknown).
3. *Sample size* – only patients meeting ≥2 Rotterdam criteria receive a phenotype; the rest are excluded and counted below.

In [ ]:
CONFIG = {
    "ha_use_hirsutism": True,
    "ha_use_acne": False,
    "ha_use_skin_darkening": False,
    "use_cycle_length_rule": True,       # auto-disabled if the column cannot support it
    "cycle_length_threshold": 35,
    "irregular_cycle_codes": [4],
    "regular_cycle_codes": [2],
    "pcom_rule": "sum",                  # "sum": L+R >= threshold (SOP) | "max": either ovary >= threshold
    "follicle_threshold": 20,
}

PHENOTYPE_NAMES = {
    0: "A (Classic): HA + OD + PCOM",
    1: "B: HA + OD + no PCOM",
    2: "C: HA + no OD + PCOM",
    3: "D: no HA + OD + PCOM",
}
SHORT = {0: "A", 1: "B", 2: "C", 3: "D"}
CLASS_LABELS = [0, 1, 2, 3]


def normalize_name(name):
    return re.sub(r"\s+", " ", str(name).strip().lower())


def find_column(frame, patterns, exclude_patterns=None):
    exclude_patterns = exclude_patterns or []
    for col in frame.columns:
        n = normalize_name(col)
        if any(re.search(p, n) for p in patterns) and not any(re.search(p, n) for p in exclude_patterns):
            return col
    return None


def resolve_columns(frame):
    patterns = {
        "hirsutism": [r"hair growth", r"hirsutism", r"excess hair"],
        "acne": [r"pimple", r"acne"],
        "skin_darkening": [r"skin darkening", r"darkening"],
        "cycle_length": [r"cycle length"],
        "cycle_regularity": [r"cycle\s*\(\s*r\s*/\s*i\s*\)", r"cycle.*regular", r"cycle.*irregular"],
        "follicles_left": [r"follicle no\.?\s*\(?l\)?$", r"follicle.*left"],
        "follicles_right": [r"follicle no\.?\s*\(?r\)?$", r"follicle.*right"],
    }
    resolved = {}
    for canon, pats in patterns.items():
        col = find_column(frame, pats)
        if col is not None:
            resolved[canon] = col
    return resolved


resolved = resolve_columns(df)
print("Resolved columns:")
for k, v in resolved.items():
    print(f"  {k:18s} -> {v!r}")
missing = [k for k in ["hirsutism", "cycle_regularity", "follicles_left", "follicles_right"] if k not in resolved]
if missing:
    raise ValueError(f"Required columns not resolved: {missing}")


def validate_config(cfg, frame, resolved, verbose=True):
    """Disable rules the data cannot support."""
    cfg = dict(cfg)
    if cfg["use_cycle_length_rule"]:
        if "cycle_length" not in resolved:
            cfg["use_cycle_length_rule"] = False
        else:
            mx = pd.to_numeric(frame[resolved["cycle_length"]], errors="coerce").max()
            if mx <= cfg["cycle_length_threshold"]:
                if verbose:
                    print(f"WARNING: max of '{resolved['cycle_length']}' is {mx} (<= {cfg['cycle_length_threshold']}); "
                          "cycle-length rule can never fire -> disabled.")
                cfg["use_cycle_length_rule"] = False
    return cfg


def used_columns(resolved, cfg):
    """Original columns that the phenotype rules actually read (=> label-leakage risk if used as features)."""
    cols = []
    if cfg["ha_use_hirsutism"]: cols.append(resolved["hirsutism"])
    if cfg["ha_use_acne"]: cols.append(resolved["acne"])
    if cfg["ha_use_skin_darkening"]: cols.append(resolved["skin_darkening"])
    if cfg["use_cycle_length_rule"]: cols.append(resolved["cycle_length"])
    cols.append(resolved["cycle_regularity"])
    cols += [resolved["follicles_left"], resolved["follicles_right"]]
    return cols


def tri_state(positive, negative):
    """True where `positive`, False where `negative`, NaN otherwise."""
    out = pd.Series(np.nan, index=positive.index, dtype="object")
    out[negative] = False
    out[positive] = True
    return out


def binary_flag(series):
    num = pd.to_numeric(series, errors="coerce")
    bad = num.notna() & ~num.isin([0, 1])
    if bad.any():
        print("Warning: unexpected binary codes ->", sorted(num[bad].unique().tolist()))
    return tri_state(num == 1, num == 0)


def combine_or(*flags):
    frame = pd.concat(flags, axis=1)
    any_true = frame.eq(True).any(axis=1)
    any_known = frame.notna().any(axis=1)
    out = pd.Series(np.nan, index=frame.index, dtype="object")
    out[any_known & ~any_true] = False
    out[any_true] = True
    return out


def derive_ha(frame, resolved, cfg):
    flags = []
    if cfg["ha_use_hirsutism"]: flags.append(binary_flag(frame[resolved["hirsutism"]]))
    if cfg["ha_use_acne"]: flags.append(binary_flag(frame[resolved["acne"]]))
    if cfg["ha_use_skin_darkening"]: flags.append(binary_flag(frame[resolved["skin_darkening"]]))
    if not flags:
        raise ValueError("No hyperandrogenism indicator enabled.")
    return combine_or(*flags)


def derive_od(frame, resolved, cfg):
    flags = []
    if cfg["use_cycle_length_rule"]:
        cl = pd.to_numeric(frame[resolved["cycle_length"]], errors="coerce")
        thr = cfg["cycle_length_threshold"]
        flags.append(tri_state(cl > thr, cl <= thr))
    cr = pd.to_numeric(frame[resolved["cycle_regularity"]], errors="coerce")
    flags.append(tri_state(cr.isin(cfg["irregular_cycle_codes"]), cr.isin(cfg["regular_cycle_codes"])))
    return combine_or(*flags)


def derive_pcom(frame, resolved, cfg):
    left = pd.to_numeric(frame[resolved["follicles_left"]], errors="coerce")
    right = pd.to_numeric(frame[resolved["follicles_right"]], errors="coerce")
    thr = cfg["follicle_threshold"]
    if cfg["pcom_rule"] == "sum":
        pos = (left + right) >= thr
    elif cfg["pcom_rule"] == "max":
        pos = np.maximum(left, right) >= thr
    else:
        raise ValueError("pcom_rule must be 'sum' or 'max'.")
    complete = left.notna() & right.notna()
    return tri_state(pos & complete, ~pos & complete)


def derive_rotterdam_phenotypes(frame, resolved, cfg):
    out = frame.copy()
    out["hyperandrogenism"] = derive_ha(out, resolved, cfg)
    out["ovulatory_dysfunction"] = derive_od(out, resolved, cfg)
    out["pcom"] = derive_pcom(out, resolved, cfg)

    ha, od, pc = out["hyperandrogenism"], out["ovulatory_dysfunction"], out["pcom"]
    out["criteria_count"] = pd.concat([ha, od, pc], axis=1).eq(True).sum(axis=1)
    complete = ha.notna() & od.notna() & pc.notna()

    def m(h, o, p):
        return complete & ha.eq(h) & od.eq(o) & pc.eq(p)

    out["phenotype"] = -1                                   # unknown / incomplete
    out.loc[m(True, True, True), "phenotype"] = 0           # A
    out.loc[m(True, True, False), "phenotype"] = 1          # B
    out.loc[m(True, False, True), "phenotype"] = 2          # C
    out.loc[m(False, True, True), "phenotype"] = 3          # D
    out.loc[complete & (out["phenotype"] == -1), "phenotype"] = -2   # < 2 criteria

    names = {**PHENOTYPE_NAMES, -1: "Unknown / incomplete criteria", -2: "Fewer than two criteria"}
    out["phenotype_name"] = out["phenotype"].map(names)
    out["phenotype_status"] = np.where(out["phenotype"] >= 0, "eligible_rotterdam_phenotype",
                              np.where(out["phenotype"] == -2, "fewer_than_two_criteria", "unknown_or_incomplete"))
    return out


CFG = validate_config(CONFIG, df, resolved)
phenotyped_df = derive_rotterdam_phenotypes(df, resolved, CFG)

print("\nPhenotype counts (all patients):")
print(phenotyped_df["phenotype_name"].value_counts())
print(f"\nExcluded from modelling: {(phenotyped_df['phenotype'] < 0).sum()} of {len(phenotyped_df)} patients")

# --- Sensitivity of the phenotype counts to the proxy definitions (report this in the thesis) ---
SCENARIOS = {
    "Primary (HA = hirsutism; PCOM = L+R >= 20)": {},
    "Broad HA (hirsutism | acne | skin darkening)": {"ha_use_acne": True, "ha_use_skin_darkening": True},
    "PCOM = either ovary >= 12": {"pcom_rule": "max", "follicle_threshold": 12},
    "PCOM = either ovary >= 20": {"pcom_rule": "max", "follicle_threshold": 20},
}
rows = {}
for label, over in SCENARIOS.items():
    cfg = validate_config({**CONFIG, **over}, df, resolved, verbose=False)
    res = derive_rotterdam_phenotypes(df, resolved, cfg)["phenotype"].value_counts()
    rows[label] = {**{SHORT[k]: int(res.get(k, 0)) for k in CLASS_LABELS},
                   "<2 criteria": int(res.get(-2, 0)), "unknown": int(res.get(-1, 0))}
sensitivity_df = pd.DataFrame(rows).T
sensitivity_df["modelled_n"] = sensitivity_df[["A", "B", "C", "D"]].sum(axis=1)
print("\nSensitivity of phenotype counts to proxy definitions:")
print(sensitivity_df)

## 3. Feature engineering and a leakage-safe feature set

The phenotype label is a deterministic function of hirsutism, cycle regularity and follicle counts.
If those columns (or anything computed from them, e.g. total follicle count) are used as predictors, the model just re-learns the rules
and SHAP explains the rules rather than clinical signal. So:
* **`X_clean` (primary):** rule inputs, label columns and `total_follicles` removed.
* **`X_leaky` (diagnostic only):** used once below to *demonstrate* the inflated performance.

In [ ]:
def engineer_features(frame):
    d = frame.copy()
    w = find_column(d, [r"^weight\s*\(kg"]); h = find_column(d, [r"^height"])
    fsh = find_column(d, [r"^fsh\s*\("]); lh = find_column(d, [r"^lh\s*\("])
    waist = find_column(d, [r"^waist\s*\(inch"]); hip = find_column(d, [r"^hip\s*\(inch"])
    fl, fr = resolved["follicles_left"], resolved["follicles_right"]
    sl = find_column(d, [r"avg\.?\s*f size \(l\)"]); sr = find_column(d, [r"avg\.?\s*f size \(r\)"])

    if w and h:
        d["BMI_calc"] = d[w] / ((d[h] / 100.0) ** 2)
    if fsh and lh:
        d["FSH_LH_ratio"] = d[fsh] / d[lh].where(d[lh] > 0)          # LH/FSH is the reciprocal
    if waist and hip:
        d["Waist_Hip_ratio"] = d[waist] / d[hip].where(d[hip] > 0)
    d["total_follicles"] = d[fl] + d[fr]                              # derived from rule inputs -> leakage
    if sl and sr:
        d["mean_follicle_size_mm"] = d[[sl, sr]].mean(axis=1)

    # Drop the dataset's pre-computed versions that were just re-derived
    originals = [find_column(d, [r"^bmi$"]), find_column(d, [r"^fsh/lh$"]), find_column(d, [r"waist\s*:\s*hip"])]
    d = d.drop(columns=[c for c in originals if c], errors="ignore")
    return d.replace([np.inf, -np.inf], np.nan)


model_df = phenotyped_df[phenotyped_df["phenotype"].isin(CLASS_LABELS)].reset_index(drop=True)
model_df = engineer_features(model_df)

LABEL_COLS = ["phenotype", "phenotype_name", "phenotype_status", "hyperandrogenism",
              "ovulatory_dysfunction", "pcom", "criteria_count"]
RULE_INPUT_COLS = [c for c in used_columns(resolved, CFG) if c in model_df.columns]
DERIVED_LEAK_COLS = ["total_follicles"]

excluded = set(LABEL_COLS + RULE_INPUT_COLS + DERIVED_LEAK_COLS)
feature_cols_clean = [c for c in model_df.columns
                      if c not in excluded and pd.api.types.is_numeric_dtype(model_df[c])]
feature_cols_leaky = feature_cols_clean + RULE_INPUT_COLS + DERIVED_LEAK_COLS

X_clean = model_df[feature_cols_clean].copy()
X_leaky = model_df[feature_cols_leaky].copy()
y = model_df["phenotype"].astype(int).values

print(f"Modelling dataset: {len(model_df)} patients | clean features: {len(feature_cols_clean)}")
print("Class counts:", {SHORT[k]: int((y == k).sum()) for k in CLASS_LABELS})
print("Excluded from X_clean (leakage guard):", sorted(set(RULE_INPUT_COLS + DERIVED_LEAK_COLS)))
print("Clean features:", feature_cols_clean)

## 4. Preprocessing, models and cross-validation utilities
Median imputation → |Z|>3 capping (continuous columns only) → StandardScaler → SMOTE (k=3), all **fitted on training folds only**.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (f1_score, cohen_kappa_score, matthews_corrcoef, accuracy_score,
                             classification_report, confusion_matrix)
from imblearn.over_sampling import SMOTE


class ZScoreCapper(BaseEstimator, TransformerMixin):
    """Cap continuous columns at mean ± z*std (binary columns are left untouched)."""
    def __init__(self, z=3.0):
        self.z = z

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.mean_ = np.nanmean(X, axis=0)
        std = np.nanstd(X, axis=0)
        self.std_ = np.where(std == 0, 1.0, std)
        self.continuous_ = np.array([len(np.unique(X[~np.isnan(X[:, j]), j])) > 2 for j in range(X.shape[1])])
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        lo, hi = self.mean_ - self.z * self.std_, self.mean_ + self.z * self.std_
        for j in np.where(self.continuous_)[0]:
            X[:, j] = np.clip(X[:, j], lo[j], hi[j])
        return X


def make_preprocessor():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("cap", ZScoreCapper(z=3.0)),
        ("scale", StandardScaler()),
    ])


def apply_smote(X, y_, k=3):
    min_count = np.bincount(y_).min()
    k_eff = max(1, min(k, min_count - 1))
    return SMOTE(k_neighbors=k_eff, random_state=RANDOM_STATE).fit_resample(X, y_)


# --- TabNet (installed in the setup cell) ---
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    RUN_TABNET = True
except Exception:
    RUN_TABNET = False
    print("TabNet not available -> skipped. Re-run the install cell (and restart runtime) to enable it.")


class TabNetSklearn:
    def fit(self, X, y_):
        self.m = TabNetClassifier(seed=RANDOM_STATE, verbose=0)
        self.m.fit(X.astype(np.float32), y_, max_epochs=100, batch_size=32,
                   virtual_batch_size=16, drop_last=False)
        return self

    def predict(self, X):
        return self.m.predict(X.astype(np.float32))

    @property
    def feature_importances_(self):
        return self.m.feature_importances_


def build_xgb():
    from xgboost import XGBClassifier
    return XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, subsample=0.8,
                         colsample_bytree=0.8, objective="multi:softprob", eval_metric="mlogloss",
                         random_state=RANDOM_STATE, n_jobs=-1)


def build_lgbm():
    from lightgbm import LGBMClassifier
    # min_child_samples / num_leaves lowered because the modelling set is small (~110 patients)
    return LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=15, min_child_samples=5,
                          class_weight="balanced", random_state=RANDOM_STATE, verbose=-1)


def build_rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                  random_state=RANDOM_STATE, n_jobs=-1)


MODELS = {   # SOP: SMOTE for XGBoost; class weights for LightGBM / Random Forest
    "XGBoost (primary)": {"build": build_xgb, "smote": True},
    "LightGBM": {"build": build_lgbm, "smote": False},
    "Random Forest": {"build": build_rf, "smote": False},
}
if RUN_TABNET:
    MODELS["TabNet"] = {"build": TabNetSklearn, "smote": True}


def fit_predict(name, X_tr, y_tr, X_te):
    prep = make_preprocessor()
    Xtr = prep.fit_transform(X_tr)
    Xte = prep.transform(X_te)
    ytr = y_tr
    if MODELS[name]["smote"]:
        Xtr, ytr = apply_smote(Xtr, y_tr, k=3)
    model = MODELS[name]["build"]().fit(Xtr, ytr)
    return np.asarray(model.predict(Xte)).astype(int)


def score(y_true, y_pred):
    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro", labels=CLASS_LABELS, zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", labels=CLASS_LABELS, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
    }


CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)


def evaluate_model(name, X, y_):
    fold_scores, all_true, all_pred = [], [], []
    for tr, te in CV.split(X, y_):
        pred = fit_predict(name, X.iloc[tr], y_[tr], X.iloc[te])
        fold_scores.append(score(y_[te], pred))
        all_true.append(y_[te]); all_pred.append(pred)
    return pd.DataFrame(fold_scores), np.concatenate(all_true), np.concatenate(all_pred)


def summarise(fold_df):
    return {m: f"{fold_df[m].mean():.3f} ± {fold_df[m].std():.3f}" for m in fold_df.columns}

## 5. Benchmark: XGBoost vs LightGBM vs Random Forest (vs TabNet)
Repeated (3×) stratified 5-fold CV. With only ~110 patients, report mean ± SD and treat differences between models cautiously.

In [ ]:
results, pooled = {}, {}
for name in MODELS:
    print("Evaluating", name, "...")
    fold_df, yt, yp = evaluate_model(name, X_clean, y)
    results[name] = summarise(fold_df)
    pooled[name] = (yt, yp)

results_df = pd.DataFrame(results).T
print("\n=== Benchmark on X_clean (mean ± SD over 15 folds) ===")
print(results_df)

primary = "XGBoost (primary)"
yt, yp = pooled[primary]
print(f"\nPer-class report for {primary} (pooled out-of-fold predictions; recall = sensitivity):")
print(classification_report(yt, yp, labels=CLASS_LABELS,
                            target_names=[PHENOTYPE_NAMES[k] for k in CLASS_LABELS], digits=3, zero_division=0))

cm = confusion_matrix(yt, yp, labels=CLASS_LABELS)
fig, ax = plt.subplots(figsize=(5, 4.2))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels([SHORT[k] for k in CLASS_LABELS]); ax.set_yticklabels([SHORT[k] for k in CLASS_LABELS])
ax.set_xlabel("Predicted phenotype"); ax.set_ylabel("Derived phenotype"); ax.set_title(f"{primary} — pooled CV confusion matrix")
for i in range(4):
    for j in range(4):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix_xgboost.png", dpi=200); plt.show()

## 6. Leakage check (diagnostic — do **not** report this as model performance)

In [ ]:
leaky_df, _, _ = evaluate_model(primary, X_leaky, y)
clean_df, _, _ = evaluate_model(primary, X_clean, y)
print(f"XGBoost macro-F1 with rule-input features (leaky): {leaky_df['macro_f1'].mean():.3f}")
print(f"XGBoost macro-F1 on clean features (primary)    : {clean_df['macro_f1'].mean():.3f}")
print("A large gap means the model re-learns the labelling rules; the clean number is the honest one.")

## 7. Explainability — TreeSHAP (global and per-phenotype local)
The explanation model is fitted once on all patients (SMOTE-balanced) purely for interpretation; performance numbers above come from CV.

In [ ]:
import shap

prep_full = make_preprocessor()
Xs = prep_full.fit_transform(X_clean)
Xs_res, y_res = apply_smote(Xs, y, k=3)
xgb_full = build_xgb().fit(Xs_res, y_res)

X_scaled_df = pd.DataFrame(Xs, columns=feature_cols_clean)
X_display = X_clean.fillna(X_clean.median())          # unscaled values, for readable waterfall plots

explainer = shap.TreeExplainer(xgb_full)
sv = explainer.shap_values(X_scaled_df)
sv = np.stack(sv, axis=-1) if isinstance(sv, list) else np.asarray(sv)   # -> (patients, features, classes)
assert sv.shape == (len(X_scaled_df), len(feature_cols_clean), len(CLASS_LABELS)), sv.shape

# Global importance (mean |SHAP| over patients and classes)
global_imp = pd.Series(np.abs(sv).mean(axis=(0, 2)), index=feature_cols_clean).sort_values(ascending=False)
top = global_imp.head(15)[::-1]
plt.figure(figsize=(7, 5)); plt.barh(top.index, top.values)
plt.xlabel("Mean |SHAP value|"); plt.title("Global feature importance — XGBoost (all phenotypes)")
plt.tight_layout(); plt.savefig("shap_global_bar.png", dpi=200); plt.show()

# Top drivers per phenotype
per_class = {SHORT[k]: pd.Series(np.abs(sv[:, :, k]).mean(axis=0), index=feature_cols_clean)
             .sort_values(ascending=False).head(5).index.tolist() for k in CLASS_LABELS}
print("Top-5 SHAP features per phenotype:")
for k, v in per_class.items():
    print(f"  {k}: {v}")

# Alignment of SHAP top features with clinical literature (SOP §5.5, XAI quality)
LITERATURE_EXPECTED = {"AMH": "amh", "LH / FSH-LH ratio": "lh", "BMI": "bmi", "Waist:hip ratio": "waist_hip",
                       "Follicle size": "follicle_size"}
top10 = [f.lower() for f in global_imp.head(10).index]
alignment = {label: any(key in f for f in top10) for label, key in LITERATURE_EXPECTED.items()}
print("\nLiterature-expected drivers present in SHAP top-10:", alignment)

# Local waterfall: the most confidently predicted patient of each phenotype
proba = xgb_full.predict_proba(Xs)
base = np.atleast_1d(explainer.expected_value)
for k in CLASS_LABELS:
    idx = np.where(y == k)[0]
    i = idx[np.argmax(proba[idx, k])]
    exp = shap.Explanation(values=sv[i, :, k], base_values=float(base[k] if len(base) > 1 else base[0]),
                           data=X_display.iloc[i].values, feature_names=feature_cols_clean)
    shap.plots.waterfall(exp, max_display=10, show=False)
    plt.title(f"Phenotype {SHORT[k]} — patient row {i} (P={proba[i, k]:.2f})")
    plt.tight_layout(); plt.savefig(f"shap_waterfall_{SHORT[k]}.png", dpi=200, bbox_inches="tight"); plt.show()

# TabNet attention-based importance (alternative to SHAP), if available
if RUN_TABNET:
    tn = TabNetSklearn().fit(Xs_res, y_res)
    print("\nTabNet top-10 features:")
    print(pd.Series(tn.feature_importances_, index=feature_cols_clean).sort_values(ascending=False).head(10))

## 8. Unified decision-support mapping (Module 2 half of SOP §5.4)
Text copied from the SOP table. This is a research prototype, not clinical advice — have the supervisor/clinician verify it against current guidelines.

In [ ]:
PHENOTYPE_ACTIONS = {
    0: "Aggressive metabolic screening (HOMA-IR, lipids, HbA1c); metformin + OCP; lifestyle intervention (5-10% weight-loss target).",
    1: "Metformin for insulin resistance; OCPs for hyperandrogenism; clomiphene/letrozole for fertility; no PCOM-specific intervention needed.",
    2: "Spironolactone for hirsutism/acne; OCPs optional; fertility often preserved; metabolic screening warranted.",
    3: "No first-line metformin; no anti-androgens; letrozole for ovulation induction; cyclical progestins for endometrial protection.",
}


def decision_support(module1_output, phenotype=None):
    """module1_output: 'PCOS', 'Normal' or 'Inconclusive' (from the image module); phenotype: 0-3 (from this module)."""
    if module1_output == "Normal":
        return "Routine gynecological follow-up; no PCOS-specific treatment."
    if module1_output == "Inconclusive":
        return "Repeat ultrasound or add clinical workup (hormonal panel, glucose tolerance test)."
    if phenotype not in PHENOTYPE_ACTIONS:
        raise ValueError("A phenotype (0-3) is required when the image module reports PCOS.")
    return f"Phenotype {SHORT[phenotype]}: {PHENOTYPE_ACTIONS[phenotype]}"


print(decision_support("PCOS", phenotype=0))

## 9. Save artifacts

In [ ]:
results_df.to_csv("module2_benchmark_results.csv")
sensitivity_df.to_csv("phenotype_sensitivity_analysis.csv")
phenotyped_df.to_csv("phenotype_derivation_audit.csv", index=False)
model_df.to_csv("phenotyped_model_dataset.csv", index=False)
with open("phenotype_rule_config.json", "w") as f:
    json.dump({"config": CFG, "resolved_columns": resolved, "features_clean": feature_cols_clean,
               "excluded_leakage_columns": sorted(set(RULE_INPUT_COLS + DERIVED_LEAK_COLS))}, f, indent=2)
print("Saved to", os.getcwd())

# Download everything (Colab) — otherwise files vanish when the runtime resets
try:
    from google.colab import files as colab_files
    out_files = [f for f in os.listdir(".") if f.endswith((".csv", ".json", ".png"))]
    import zipfile
    with zipfile.ZipFile("module2_outputs.zip", "w") as z:
        for f in out_files:
            z.write(f)
    colab_files.download("module2_outputs.zip")
except ImportError:
    pass